In [6]:
import pyspark
from pyspark import SparkContext
import imageio
import os
import numpy as np
import time

In [1]:
def readImg(path):
    img = imageio.imread(path)
    im = np.array(img, dtype='uint8')
    return im

def writeImg(path, buf):
    imageio.imwrite(path, buf)

def part_median_filter(local_data):
    part_id = local_data[0]
    first   = local_data[1]
    end     = local_data[2]
    buf     = local_data[3]
    nx = buf.shape[0]
    ny = buf.shape[1]
    
    # Initialize the new buffer for the filtered image (with three color channels)
    new_buf = np.zeros((end - first, ny, 3), dtype='uint8')

    # Helper function to get the pixel value in the neighborhood
    def get_neighbors(x, y, channel):
        neighbors = []
        for i in range(-1, 2):
            for j in range(-1, 2):
                xi = min(max(x + i, 0), nx - 1)
                yi = min(max(y + j, 0), ny - 1)
                neighbors.append(buf[xi, yi, channel])
        return np.median(neighbors)

    # Apply the median filter to each pixel and each channel (R, G, B)
    for i in range(first, end):
        for j in range(ny):
            for c in range(3):  # For each channel: 0 -> Red, 1 -> Green, 2 -> Blue
                new_buf[i - first, j, c] = get_neighbors(i, j, c)
    
    return part_id, new_buf

In [8]:
def main():
    data_dir = 'data'
    file = os.path.join(data_dir, 'lena_noisy.jpg')
    img_buf = readImg(file)
    print('SHAPE', img_buf.shape)
    nx = img_buf.shape[0]
    ny = img_buf.shape[1]

    # Split images into nb_partitions parts
    nb_partitions = 8
    print("NB PARTITIONS : ", nb_partitions)
    data = []
    begin = 0
    block_size = nx // nb_partitions
    for ip in range(nb_partitions):
        end = min(begin + block_size, nx)
        data.append([ip, begin, end, img_buf])
        begin = end

    # Create SparkContext
    sc = SparkContext()
    data_rdd = sc.parallelize(data, nb_partitions)

    # Parallel median filter computation
    start_time = time.time()
    result_rdd = data_rdd.map(part_median_filter)
    result_data = result_rdd.collect()
    end_time = time.time()
    print(f'Execution Time is : {end_time - start_time} seconds')

    # Reconstruct the new image from the result data
    new_img_buf = np.zeros((nx, ny, 3), dtype='uint8')  # Image with 3 channels (R, G, B)
    for part_id, part_buf in result_data:
        first = data[part_id][1]
        end = data[part_id][2]
        new_img_buf[first:end, :, :] = part_buf

    # Write the filtered image
    print('CREATE NEW PICTURE FILE')
    filter_file = os.path.join(data_dir, 'lena_filter_spark.jpg')
    writeImg(filter_file, new_img_buf)
    print('IMAGE CREATED SUCCESSFULLY !')
    sc.stop()

if __name__ == '__main__':
    main()

/tmp/ipykernel_938/2366238986.py:2: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img = imageio.imread(path)


SHAPE (128, 128, 3)
NB PARTITIONS :  8
Execution Time is : 0.5420904159545898 seconds
CREATE NEW PICTURE FILE
IMAGE CREATED SUCCESSFULLY !
